# Chapter 6 — Variational Autoencoders

Chapter 5 ended in grey fog. We had a decoder that turns 32 numbers into a
garment, and a cloud of real codes with a known mean and spread, and drawing
fresh numbers from that mean and spread produced structureless mush. The
diagnosis was precise:

> It is not enough to map data into a latent space. You need a latent space you
> know how to **sample from**, one where the distribution you draw from and
> the distribution the decoder was trained on are the *same distribution*.

The **variational autoencoder** is the first of the three answers to that, and
the most direct one: if the problem is that the latent distribution is an
unknown lumpy cloud, then *make it* a distribution you already know. Add a term
to the loss that pulls the codes toward a standard Gaussian, and sampling
becomes `np.random.normal`.

That single change costs one new piece of machinery (the reparameterization
trick, so gradients can flow through a random draw) and buys the first working
generative model in this curriculum.

| Module | What you build | Dataset (Hugging Face) |
|---|---|---|
| 1 | Chapter 5's plain autoencoder and its failed sampling, reproduced | `zalando-datasets/fashion_mnist` |
| 2 | The **VAE**: fuzzy codes, the KL term, the reparameterization trick, a custom `train_step` | `zalando-datasets/fashion_mnist` |
| 3 | A **denoising** autoencoder, the seed of Chapter 8 | `zalando-datasets/fashion_mnist` |

**How each concept is presented**, the same three passes as earlier chapters:

> 🧠 **The intuition:** the idea in plain language, no symbols.
> 📐 **The math:** the same idea written precisely, so you can read papers.
> 💻 **The code:** the same idea again, executable, in the cell that follows.

**Runtime:** ~3 minutes on a GPU, ~15 on CPU, for two trainings: the plain
autoencoder baseline in Module 1 and the VAE in Module 2. Every training cell has a knob
marked `# <- knob`. Fashion-MNIST is already cached from Chapter 5.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from datasets import load_dataset
from sklearn.decomposition import PCA

keras.utils.set_random_seed(42)
rng = np.random.default_rng(seed=42)
plt.rcParams["figure.figsize"] = (7, 4.5)

print("TensorFlow", tf.__version__, "| Keras", keras.__version__)
print("GPU:", tf.config.list_physical_devices("GPU") or "none, so CPU only, which still works")

---
# Module 1 — Chapter 5's Failure, Reproduced

💻 Nothing new here, and it is worth running rather than skipping: this is
Chapter 5's data, its encoder/decoder builders, and its plain autoencoder,
retrained so that the **grey fog** is sitting in front of us as a concrete
baseline rather than a memory. Module 2 has to beat the last figure in this
module.

(Same seed, same data, same architecture as Chapter 5, so this is that
experiment run again rather than a new one.)


In [ ]:
fashion = load_dataset("zalando-datasets/fashion_mnist")   # ~30 MB on first run, then cached
class_names = fashion["train"].features["label"].names
print(fashion)
print(class_names)


def split_to_arrays(split, n=None, image_col="image", label_col="label", seed=42):
    """Shuffle a HF split, optionally take the first n rows, return (images, labels)."""
    ds = split.shuffle(seed=seed)
    if n is not None:
        ds = ds.select(range(n))
    images = np.stack([np.array(im) for im in ds[image_col]])
    labels = np.array(ds[label_col])
    return images, labels


N_TRAIN = 20_000                                    # <- knob: up to 60_000
X_train_raw, y_train = split_to_arrays(fashion["train"], n=N_TRAIN)
X_test_raw,  y_test  = split_to_arrays(fashion["test"])

# Autoencoders want inputs in [0, 1] so a sigmoid output layer can match them exactly.
X_train = (X_train_raw / 255.0).astype("float32")[..., None]   # (N, 28, 28, 1)
X_test  = (X_test_raw  / 255.0).astype("float32")[..., None]

print("train:", X_train.shape, "| test:", X_test.shape, "| range:", X_train.min(), X_train.max())

In [ ]:
# --- Chapter 5's encoder and decoder, verbatim ------------------------------
def build_encoder(latent_dim):
    inputs = keras.Input(shape=(28, 28, 1))
    x = layers.Conv2D(32, 3, padding="same", activation="relu")(inputs)   # 28x28x32
    x = layers.MaxPooling2D(2)(x)                                         # 14x14x32
    x = layers.Conv2D(64, 3, padding="same", activation="relu")(x)        # 14x14x64
    x = layers.MaxPooling2D(2)(x)                                         #  7x7x64
    x = layers.Flatten()(x)                                               # 3136
    code = layers.Dense(latent_dim, name="code")(x)                       # the bottleneck
    return keras.Model(inputs, code, name="encoder")


def build_decoder(latent_dim):
    codes = keras.Input(shape=(latent_dim,))
    x = layers.Dense(7 * 7 * 64, activation="relu")(codes)
    x = layers.Reshape((7, 7, 64))(x)                                                   #  7x7x64
    x = layers.Conv2DTranspose(64, 3, strides=2, padding="same", activation="relu")(x)  # 14x14x64
    x = layers.Conv2DTranspose(32, 3, strides=2, padding="same", activation="relu")(x)  # 28x28x32
    out = layers.Conv2D(1, 3, padding="same", activation="sigmoid")(x)                  # 28x28x1
    return keras.Model(codes, out, name="decoder")


pick = rng.choice(len(X_test), size=8, replace=False)   # one fixed sample, reused in every figure
print("encoder/decoder builders ready | fixed sample:", pick)

In [ ]:
# --- Chapter 5's plain autoencoder, retrained so we have something to beat ----
LATENT_DIM = 32
encoder, decoder = build_encoder(LATENT_DIM), build_decoder(LATENT_DIM)

inputs = keras.Input(shape=(28, 28, 1))
autoencoder = keras.Model(inputs, decoder(encoder(inputs)), name="conv_autoencoder")
autoencoder.compile(optimizer=keras.optimizers.Adam(1e-3), loss="mse")
autoencoder.fit(X_train, X_train, validation_data=(X_test, X_test),
                epochs=20, batch_size=128, verbose=2)          # <- knob

# Chapter 5, Module 3.3, repeated: draw latent vectors using the codes' own
# per-axis mean and spread, and decode them. This produced grey fog.
codes_test = encoder.predict(X_test, verbose=0)
z_random = rng.normal(loc=codes_test.mean(axis=0),
                      scale=codes_test.std(axis=0),
                      size=(8, LATENT_DIM)).astype("float32")
fake = decoder.predict(z_random, verbose=0)

fig, axes = plt.subplots(1, 8, figsize=(12, 1.8))
for ax, img in zip(axes, fake):
    ax.imshow(img.squeeze(), cmap="gray", vmin=0, vmax=1)
    ax.axis("off")
fig.suptitle("The failure this chapter has to fix: random latent vectors, decoded", y=1.12)
plt.show()

---
# Module 2 — The Variational Autoencoder

🧠 **The intuition.** The plain autoencoder gave each image a single *pin* on
the map, and everywhere between the pins was undefined territory. The VAE
changes two things about that.

First, each image no longer gets a pin. It gets a **fuzzy blob**. During
training we don't decode the exact code; we decode a random point *near* it. So
the decoder is forced to produce a sensible garment for a whole neighbourhood,
not one point. Blobs of nearby images overlap, and the holes close up.

Second, we add a gentle force pulling every blob **toward the origin, at unit
size**. Left alone, the encoder would push blobs infinitely far apart to keep
them from being confused. Instead they get crowded into one tidy region shaped
like a standard bell curve, and *that* is a shape we know how to roll dice
from. The two forces fight: reconstruction wants the blobs spread out and
distinct, the regularizer wants them merged at the origin. The compromise is a
latent space that is both meaningful and samplable.

📐 **The math.** The encoder now outputs a distribution per image,
$q_\phi(z \mid x) = \mathcal{N}\!\big(\mu_\phi(x),\, \mathrm{diag}\,\sigma^2_\phi(x)\big)$,
and we minimise

$$ \mathcal{L} \;=\; \underbrace{\mathbb{E}_{z \sim q_\phi(z|x)}\big[-\log p_\theta(x \mid z)\big]}_{\text{reconstruction: spread the blobs apart}} \;+\; \beta \underbrace{D_{\mathrm{KL}}\!\big(q_\phi(z \mid x)\,\|\,p(z)\big)}_{\text{regularizer: pull them to } \mathcal{N}(0, I)} $$

(This is the negative **ELBO**, a lower bound on the data's log-likelihood,
which is where the "variational" in the name comes from.) For diagonal
Gaussians the KL term has a closed form with no sampling needed:

$$ D_{\mathrm{KL}} = -\tfrac{1}{2}\sum_{j=1}^{d} \left(1 + \log \sigma_j^2 - \mu_j^2 - \sigma_j^2\right) $$

Read that sum term by term: it is zero exactly when $\mu_j = 0$ and
$\sigma_j = 1$, and grows as the blob drifts off-centre or changes size.

## 2.1 The reparameterization trick

🧠 **The intuition.** There is one obstacle. Backpropagation needs to ask "if I
nudge $\mu$, how does the loss change?", but the step in between is *roll a
random number*, and you cannot differentiate a dice roll. The fix is a change
of bookkeeping: instead of "draw a sample from a blob at $\mu$ with width
$\sigma$", say "roll a *standard* random number $\varepsilon$, then shift and
scale it by $\mu$ and $\sigma$". Identical outcome, but now the randomness is
an *input* the network doesn't control, and $\mu$ and $\sigma$ sit on a plain
differentiable path.

📐 **The math.**

$$ z \sim \mathcal{N}(\mu, \sigma^2) \quad\Longleftrightarrow\quad z = \mu + \sigma \odot \varepsilon, \;\; \varepsilon \sim \mathcal{N}(0, I) $$

The left form has no usable gradient; the right form gives
$\partial z / \partial \mu = 1$ and $\partial z / \partial \sigma = \varepsilon$.
Same distribution, gradients flow. This one-line trick is what made VAEs
trainable, and the identical idea reappears in Chapter 8 as the closed-form
noising step, where it lets you jump to any noise level in O(1).

💻 **The code.** We use `latent_dim = 2` deliberately. It costs reconstruction
quality, but buys something better: a latent space you can **draw on paper**.


In [ ]:
VAE_LATENT = 2                                        # <- knob: 2 for the pictures, 16 for quality


class Sampling(layers.Layer):
    """z = mu + sigma * eps: the reparameterization trick as a layer."""

    def call(self, inputs):
        z_mean, z_log_var = inputs
        eps = tf.random.normal(shape=tf.shape(z_mean))       # the randomness enters as an INPUT
        return z_mean + tf.exp(0.5 * z_log_var) * eps        # ...so gradients reach mu and sigma


def build_vae_encoder(latent_dim):
    inputs = keras.Input(shape=(28, 28, 1))
    x = layers.Conv2D(32, 3, strides=2, padding="same", activation="relu")(inputs)  # 14x14
    x = layers.Conv2D(64, 3, strides=2, padding="same", activation="relu")(x)       #  7x7
    x = layers.Flatten()(x)
    x = layers.Dense(128, activation="relu")(x)
    z_mean = layers.Dense(latent_dim, name="z_mean")(x)          # centre of the blob
    z_log_var = layers.Dense(latent_dim, name="z_log_var")(x)    # log of its variance
    z = Sampling()([z_mean, z_log_var])
    return keras.Model(inputs, [z_mean, z_log_var, z], name="vae_encoder")


vae_encoder = build_vae_encoder(VAE_LATENT)
vae_decoder = build_decoder(VAE_LATENT)          # Module 1's decoder, narrower input
vae_encoder.summary()

*(Why `z_log_var` instead of `sigma`? A network's output is any real number,
but a variance must be positive. Predicting the **log** of it makes every
possible output legal, and `exp(0.5 * log_var)` recovers $\sigma$. This
"predict the log, exponentiate later" pattern is everywhere in ML.)*

## 2.2 A custom `train_step`

🧠 **The intuition.** `model.compile(loss=...)` assumes the loss compares the
final output to a target. Ours doesn't. It also needs $\mu$ and $\sigma$ from
the middle of the network. So we write the training step by hand: run the
forward pass while a `GradientTape` records everything, compute both loss
terms, ask the tape for gradients, hand them to the optimizer. Read it closely;
it is the same pattern you will use for the GAN in Chapter 7.

📐 **A note on the reconstruction term.** We use **binary cross-entropy summed
over pixels** rather than MSE:
$-\sum_{\text{pixels}} \left[x \log \hat{x} + (1 - x)\log(1 - \hat{x})\right]$.
On $[0,1]$ images this treats each pixel as a Bernoulli likelihood and
penalises confident mistakes far more sharply than squared error does, which
yields noticeably less mush. Summing over pixels rather than averaging keeps it
on a comparable scale to the KL term, so $\beta \approx 1$ is sensible.


In [ ]:
class VAE(keras.Model):
    def __init__(self, encoder, decoder, beta=1.0, **kwargs):
        super().__init__(**kwargs)
        self.encoder, self.decoder, self.beta = encoder, decoder, beta
        self.loss_tracker = keras.metrics.Mean(name="loss")
        self.recon_tracker = keras.metrics.Mean(name="recon")
        self.kl_tracker = keras.metrics.Mean(name="kl")

    @property
    def metrics(self):
        return [self.loss_tracker, self.recon_tracker, self.kl_tracker]

    def call(self, x):
        return self.decoder(self.encoder(x)[2])

    def compute_losses(self, x):
        z_mean, z_log_var, z = self.encoder(x)
        recon = self.decoder(z)

        # term 1: reconstruction, BCE per pixel, summed over the image, averaged over the batch
        bce = keras.losses.binary_crossentropy(x, recon)                      # (B, 28, 28)
        recon_loss = tf.reduce_mean(tf.reduce_sum(bce, axis=(1, 2)))

        # term 2: KL to N(0, I), closed form, straight from the formula above
        kl = -0.5 * tf.reduce_sum(1 + z_log_var - tf.square(z_mean) - tf.exp(z_log_var), axis=1)
        kl_loss = tf.reduce_mean(kl)

        return recon_loss + self.beta * kl_loss, recon_loss, kl_loss

    def train_step(self, data):
        x = data[0] if isinstance(data, tuple) else data
        with tf.GradientTape() as tape:                       # record the forward pass...
            loss, recon_loss, kl_loss = self.compute_losses(x)
        grads = tape.gradient(loss, self.trainable_weights)   # ...replay it backwards
        self.optimizer.apply_gradients(zip(grads, self.trainable_weights))

        self.loss_tracker.update_state(loss)
        self.recon_tracker.update_state(recon_loss)
        self.kl_tracker.update_state(kl_loss)
        return {m.name: m.result() for m in self.metrics}


vae = VAE(vae_encoder, vae_decoder, beta=1.0)
vae.compile(optimizer=keras.optimizers.Adam(1e-3),
            jit_compile=False)   # XLA miscompiles one fused matmul here on some newer GPUs;
                                 # at this scale JIT buys nothing anyway. Drop it if yours is fine.

VAE_EPOCHS = 25                                        # <- knob
vae_hist = vae.fit(X_train, epochs=VAE_EPOCHS, batch_size=128, verbose=2)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 3.6))
axes[0].plot(vae_hist.history["recon"], marker="o")
axes[0].set_title("reconstruction loss (BCE, summed over pixels)")
axes[1].plot(vae_hist.history["kl"], marker="o", color="tab:orange")
axes[1].set_title("KL divergence from $\\mathcal{N}(0, I)$")
for ax in axes:
    ax.set_xlabel("epoch"); ax.grid(True)
plt.tight_layout()
plt.show()

The KL curve is the one to watch. It rises off zero as the encoder starts
*using* the latent space, since a KL of exactly 0 would mean every image maps to
the identical blob, an encoder that has given up and told the decoder nothing (a
real failure mode called **posterior collapse**). Then it settles as the two
pressures balance. That plateau value, measured in nats, is roughly how much
information about each image the model has chosen to store.

## 2.3 Now sampling works


In [ ]:
z_prior = rng.normal(size=(8, VAE_LATENT)).astype("float32")   # straight from N(0, I)
vae_samples = vae_decoder.predict(z_prior, verbose=0)

fig, axes = plt.subplots(2, 8, figsize=(12, 3.4))
for row, batch in enumerate([fake, vae_samples]):     # row 0 = Module 1's plain-AE attempt
    for ax, img in zip(axes[row], batch):
        ax.imshow(img.squeeze(), cmap="gray", vmin=0, vmax=1)
        ax.set_xticks([]); ax.set_yticks([])
    axes[row, 0].set_ylabel(["plain AE", "VAE"][row], fontsize=10)
fig.suptitle("Same procedure: draw a random latent vector, decode it", y=1.04)
plt.tight_layout()
plt.show()

Recognisable garments, from nothing but `np.random.normal`. **This is your
first working generative model.** The regularizer did all the work: it made the
distribution we sample from and the distribution the decoder was trained on the
*same distribution*.

## 2.4 The map of everything

With `latent_dim = 2` we can do something impossible in higher dimensions:
decode a regular grid across the *entire* latent space and see the whole
learned manifold at once. We place the grid at the quantiles of the standard
normal (`norm.ppf`) because that is the distribution the space is supposed to
have, so the steps are equal in probability rather than in distance.


In [ ]:
from scipy.stats import norm

GRID = 18
grid_x = norm.ppf(np.linspace(0.005, 0.995, GRID))
grid_y = norm.ppf(np.linspace(0.005, 0.995, GRID))
z_grid = np.array([[x, y] for y in grid_y for x in grid_x], dtype="float32")

tiles = vae_decoder.predict(z_grid, verbose=0).reshape(GRID, GRID, 28, 28)
canvas = tiles.transpose(0, 2, 1, 3).reshape(GRID * 28, GRID * 28)

plt.figure(figsize=(8, 8))
plt.imshow(canvas, cmap="gray")
plt.title("The learned manifold: every point in a 2-D latent space, decoded")
plt.xlabel("$z_1$"); plt.ylabel("$z_2$")
plt.xticks([]); plt.yticks([])
plt.show()

In [ ]:
z_mean_test, _, _ = vae_encoder.predict(X_test, verbose=0)

plt.figure(figsize=(7.5, 6))
scatter = plt.scatter(z_mean_test[:, 0], z_mean_test[:, 1], c=y_test, cmap="tab10", s=4, alpha=0.6)
cbar = plt.colorbar(scatter, ticks=range(10)); cbar.ax.set_yticklabels(class_names, fontsize=8)
plt.gca().add_patch(plt.Circle((0, 0), 2, fill=False, color="k", linestyle="--", linewidth=1.5))
plt.title("VAE posterior means, with the 2σ circle of the prior")
plt.xlabel("$z_1$"); plt.ylabel("$z_2$")
plt.tight_layout()
plt.show()

Two things to notice, side by side:

- **The manifold grid is continuous.** Move a little in $z$ and the garment
  changes a little: sleeves lengthen, a boot's ankle rises, a bag squares off.
  No holes, no dead regions. This is exactly the property the plain autoencoder
  lacked in Chapter 5, Module 3.3.
- **The posterior cloud fills the dashed 2σ circle**, centred, roughly round,
  and roughly unit-scale. That is the KL term's doing, and it is *precisely* the
  condition that makes the grid above meaningful and `np.random.normal` a valid
  sampler.

The honest caveat: VAE samples are **soft**. The reconstruction term is still a
per-pixel likelihood, so when the model is unsure it hedges by blurring, the
same trap as Chapter 5, Module 2.4, and the KL term adds its own smoothing
pressure.
Chapter 7 attacks this by throwing away per-pixel losses entirely and letting a
*second network* decide what looks real.


---
# Module 3 — Denoising Autoencoders: A Deliberate Cliffhanger

🧠 **The intuition.** One last variant, because it is the seed of everything in
Chapter 8. Show the model a garment buried in static, and ask it for the clean
garment back. Now the bottleneck isn't the only teacher: to remove speckle the
model has to *know what garments look like*, so it can tell "this bright pixel
is part of a sleeve" from "this bright pixel is noise". A copier can't do that.
Denoising forces the model to learn the data distribution itself.

📐 **The math.** Corrupt the input, keep the clean target:

$$ \tilde{x} = x + \sigma \varepsilon, \quad \varepsilon \sim \mathcal{N}(0, I), \qquad \min_{\phi, \theta}\; \mathbb{E}\big\|\, x - D_\theta(E_\phi(\tilde{x})) \,\big\|^2 $$

Compare it to Chapter 5, Module 2's objective: the *only* change is a tilde on
the input.

💻 **The code.** Same encoder, same decoder, same MSE, and we just widen the
bottleneck to 64, since the model now has to carry noise-removal capacity too.


In [ ]:
NOISE_SIGMA = 0.4                                      # <- knob: try 0.1 and 0.8

X_train_noisy = np.clip(X_train + NOISE_SIGMA * rng.normal(size=X_train.shape), 0, 1).astype("float32")
X_test_noisy  = np.clip(X_test  + NOISE_SIGMA * rng.normal(size=X_test.shape),  0, 1).astype("float32")

dae_in = keras.Input(shape=(28, 28, 1))
dae = keras.Model(dae_in, build_decoder(64)(build_encoder(64)(dae_in)), name="denoising_ae")
dae.compile(optimizer=keras.optimizers.Adam(1e-3), loss="mse")
dae.fit(X_train_noisy, X_train,                        # <- noisy in, CLEAN out
        validation_data=(X_test_noisy, X_test),
        epochs=12, batch_size=128, verbose=2)          # <- knob

In [ ]:
cleaned = dae.predict(X_test_noisy[pick], verbose=0)

fig, axes = plt.subplots(3, 8, figsize=(12, 4.8))
for col, idx in enumerate(pick):
    for row, img in enumerate([X_test[idx], X_test_noisy[idx], cleaned[col]]):
        axes[row, col].imshow(img.squeeze(), cmap="gray", vmin=0, vmax=1)
        axes[row, col].set_xticks([]); axes[row, col].set_yticks([])
for row, tag in enumerate(["clean", f"+ noise σ={NOISE_SIGMA}", "denoised"]):
    axes[row, 0].set_ylabel(tag, fontsize=9)
plt.tight_layout()
plt.show()

The noise is gone and the garment is intact. Now hold two thoughts together:

1. This network, given a *noisy* image, produces a *clean* one.
2. In Chapter 5, Module 3.3 we saw that the hard part of generation is knowing
   where the data lives in a high-dimensional space.

So: what if we cranked $\sigma$ all the way up, until the input was pure static
with no garment left in it at all, and then ran the denoiser? It would have to
*invent* the garment. In one shot that's too big an ask, which is exactly why a
diffusion model doesn't ask for one shot. It trains a denoiser at **every**
noise level from "barely speckled" to "pure static", then removes a *little*
noise at a time, hundreds of times over, each step an easy one. That is
Chapter 8, and you have now built its central component.


---
# Wrap-Up

| You built | The transferable lesson |
|---|---|
| The reparameterization trick | $z = \mu + \sigma \odot \varepsilon$ moves the randomness to an *input*, so gradients reach $\mu$ and $\sigma$ |
| The KL term | Regularize the latent distribution toward a known prior and sampling becomes trivial, which is the whole VAE in one sentence |
| A custom `train_step` | When the loss needs values from the *middle* of the network, `compile(loss=...)` cannot express it; write the tape by hand |
| The 2-D manifold grid | A latent space with no holes: move a little in $z$, change the garment a little |
| Denoising autoencoder | Removing noise requires knowing the data manifold, the entire premise of diffusion |

**The blur is still there, and it is still MSE's fault.** The reconstruction
term is a per-pixel likelihood, so when the model is unsure it hedges, and the
KL term adds its own smoothing pressure. The next two chapters are two escapes:

- **Chapter 7 (GANs)** deletes the per-pixel loss entirely. A second network is
  trained to spot fakes, and the generator's only job is to fool it, a loss
  with no opinion about individual pixels, only about whether the whole image
  looks real. Sharp results, notoriously unstable training.
- **Chapter 8 (Diffusion)** keeps MSE but changes *what is predicted*: not the
  image, but the noise to remove, in hundreds of small steps. Each step is an
  easy prediction, so hedging costs almost nothing, and Module 3's denoiser is
  its central component, which is why this chapter ends there.
